# Notebook 6: Time-Series Forecasting to 2030

This notebook forecasts US food waste to landfill through 2030 using Prophet (with a linear
regression fallback), and generates sector-level trend projections.

**Outputs:**
- `outputs/ml/charts/forecast_2030.png`
- `outputs/ml/forecast_results.json`

---
## 1. Setup & Load Data

In [1]:
import os, sys
# Ensure we're at project root
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

In [2]:
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import json

os.makedirs('outputs/ml/charts', exist_ok=True)

SECTOR_COLORS = {
    'Farm': '#2C5F2D',
    'Foodservice': '#F96167',
    'Manufacturing': '#065A82',
    'Residential': '#F9E795',
    'Retail': '#028090'
}

In [3]:
# Load EDA report for national time series
with open('outputs/analysis/eda_report.json') as f:
    eda = json.load(f)

yearly = pd.DataFrame(eda['trend_2010_2024']['yearly_totals'])
yearly_fit = yearly[yearly['year'] != 2020].copy()

forecast_results = {}

print(f"Yearly data shape: {yearly.shape}")
yearly.head(3)

Yearly data shape: (15, 4)


,year,total_surplus,total_waste,total_landfill
0,2010,6.204295e+07,5.300779e+07,1.813590e+07
1,2011,6.416120e+07,5.441519e+07,1.864285e+07
2,2012,6.532155e+07,5.522973e+07,1.912097e+07


---
## 2. National Forecast (try Prophet, fallback to Linear Regression)

In [4]:
try:
    from prophet import Prophet

    ts = yearly[['year', 'total_landfill']].copy()
    ts.columns = ['ds', 'y']
    ts['ds'] = pd.to_datetime(ts['ds'], format='%Y')

    m = Prophet(yearly_seasonality=False, changepoint_prior_scale=0.5)
    m.fit(ts)
    future = m.make_future_dataframe(periods=6, freq='YE')
    fc = m.predict(future)

    forecast_national = fc[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].copy()
    forecast_national['year'] = forecast_national['ds'].dt.year
    forecast_national = forecast_national[forecast_national['year'] >= 2025]

    forecast_results['method'] = 'Prophet'
    forecast_results['national_landfill_forecast'] = []
    for _, row in forecast_national.iterrows():
        forecast_results['national_landfill_forecast'].append({
            'year': int(row['year']),
            'tons_landfill_predicted': round(row['yhat'], 0),
            'lower': round(row['yhat_lower'], 0),
            'upper': round(row['yhat_upper'], 0)
        })

    hist_years = yearly['year'].values
    hist_landfill = yearly['total_landfill'].values
    fc_years = forecast_national['year'].values
    fc_vals = forecast_national['yhat'].values
    fc_lower = forecast_national['yhat_lower'].values
    fc_upper = forecast_national['yhat_upper'].values

    print(f"Prophet forecast 2030: {fc_vals[-1]/1e6:.2f}M tons landfill")

except Exception as e:
    print(f"Prophet failed ({e}), using linear regression fallback")
    from sklearn.linear_model import LinearRegression

    X_fit = yearly_fit[['year']].values
    y_fit = yearly_fit['total_landfill'].values

    lr = LinearRegression()
    lr.fit(X_fit, y_fit)

    future_years = np.arange(2025, 2031).reshape(-1, 1)
    preds = lr.predict(future_years)

    residuals = y_fit - lr.predict(X_fit)
    se = residuals.std()

    forecast_results['method'] = 'LinearRegression (excl. 2020 COVID)'
    forecast_results['national_landfill_forecast'] = []
    for yr, pred in zip(future_years.flatten(), preds):
        forecast_results['national_landfill_forecast'].append({
            'year': int(yr),
            'tons_landfill_predicted': round(float(pred), 0),
            'lower': round(float(pred - 1.96*se), 0),
            'upper': round(float(pred + 1.96*se), 0)
        })

    hist_years = yearly['year'].values
    hist_landfill = yearly['total_landfill'].values
    fc_years = future_years.flatten()
    fc_vals = preds
    fc_lower = preds - 1.96*se
    fc_upper = preds + 1.96*se

    print(f"Linear forecast 2030: {preds[-1]/1e6:.2f}M tons landfill")

bau_2030 = forecast_results['national_landfill_forecast'][-1]['tons_landfill_predicted']
forecast_results['bau_2030_tons_landfill'] = bau_2030
forecast_results['current_2024_tons_landfill'] = round(yearly[yearly['year']==2024]['total_landfill'].values[0], 0)

print(f"BAU 2030: {bau_2030/1e6:.2f}M tons to landfill")

21:33:46 - cmdstanpy - INFO - Chain [1] start processing


21:33:47 - cmdstanpy - INFO - Chain [1] done processing


Prophet forecast 2030: 26.54M tons landfill
BAU 2030: 26.54M tons to landfill


---
## 3. Sector-Level Forecasts

In [5]:
from sklearn.linear_model import LinearRegression

sector_forecasts = {}
by_sector = eda['trend_2010_2024']['by_sector']

for sector in ['Residential', 'Foodservice', 'Retail']:
    sec_data = pd.DataFrame(by_sector[sector])
    sec_fit = sec_data[sec_data['year'] != 2020]
    lr_s = LinearRegression()
    lr_s.fit(sec_fit[['year']].values, sec_fit['landfill'].values)
    pred_2030 = lr_s.predict([[2030]])[0]
    sector_forecasts[sector] = {
        'forecast_2030': round(float(pred_2030), 0),
        'trend_slope_per_year': round(float(lr_s.coef_[0]), 0)
    }
    print(f"{sector}: 2030 forecast = {pred_2030/1e6:.2f}M tons, slope = {lr_s.coef_[0]:.0f} tons/yr")

forecast_results['sector_forecasts'] = sector_forecasts

Residential: 2030 forecast = 10.48M tons, slope = 14517 tons/yr
Foodservice: 2030 forecast = 12.17M tons, slope = 267577 tons/yr
Retail: 2030 forecast = 1.74M tons, slope = 28286 tons/yr


---
## 4. Forecast Chart

In [6]:
fig, ax = plt.subplots(figsize=(12, 7))

ax.plot(hist_years, hist_landfill/1e6, 'o-', color='#065A82', linewidth=2.5,
        markersize=6, label='Historical', zorder=5)

ax.plot(fc_years, fc_vals/1e6, 's--', color='#F96167', linewidth=2.5,
        markersize=7, label=f'Forecast ({forecast_results["method"]})', zorder=5)
ax.fill_between(fc_years, fc_lower/1e6, fc_upper/1e6, alpha=0.2, color='#F96167', label='95% CI')

ax.annotate('COVID-19\nDrop', xy=(2020, hist_landfill[hist_years==2020][0]/1e6),
            xytext=(2020.5, 16), fontsize=9, ha='left',
            arrowprops=dict(arrowstyle='->', color='gray'))

ax.annotate(f'{bau_2030/1e6:.1f}M tons\n(BAU 2030)',
            xy=(2030, bau_2030/1e6), xytext=(2028.5, bau_2030/1e6 + 1.5),
            fontsize=10, fontweight='bold', color='#F96167',
            arrowprops=dict(arrowstyle='->', color='#F96167'))

ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Tons to Landfill (Millions)', fontsize=12)
ax.set_title('US Food Waste to Landfill: Historical Trend & 2030 Forecast', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
ax.set_xlim(2009, 2031)
plt.savefig('outputs/ml/charts/forecast_2030.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print("Saved forecast_2030.png")

Saved forecast_2030.png


---
## 5. Save Results

In [7]:
with open('outputs/ml/forecast_results.json', 'w') as f:
    json.dump(forecast_results, f, indent=2)
print("Saved to outputs/ml/forecast_results.json")
print(f"BAU 2030: {bau_2030/1e6:.2f}M tons to landfill")

Saved to outputs/ml/forecast_results.json
BAU 2030: 26.54M tons to landfill
